# 010: Advanced Optimizers — Momentum, RMSProp, and Adam with full bias correction derivation

## 1. MOMENTUM (Polyak, 1964)
- **Problem**: Vanilla SGD oscillates in ravines (dimensions with high curvature).
- **Solution**: Accumulate an exponentially weighted moving average of past gradients:
  $$v_t = \beta v_{t-1} + (1 - \beta) \nabla_\theta \mathcal{L}$$
  $$\theta_{t+1} = \theta_t - \eta v_t$$
- Typical $\beta = 0.9$. Momentum smooths oscillations and accelerates convergence along consistent gradient directions.

## 2. RMSPROP (Hinton, 2012)
- **Idea**: Adapt the learning rate per-parameter by dividing by the root-mean-square of recent gradients:
  $$s_t = \beta s_{t-1} + (1 - \beta) (\nabla_\theta \mathcal{L})^2$$
  $$\theta_{t+1} = \theta_t - \frac{\eta}{\sqrt{s_t + \epsilon}} \nabla_\theta \mathcal{L}$$
- Parameters with large gradients get smaller effective learning rates (prevents explosion).

## 3. ADAM (Kingma & Ba, 2015)
- Combines Momentum + RMSProp with **bias correction**:
  $$m_t = \beta_1 m_{t-1} + (1 - \beta_1) g_t \quad \text{(first moment)}$$
  $$v_t = \beta_2 v_{t-1} + (1 - \beta_2) g_t^2 \quad \text{(second moment)}$$
- **Bias Correction**: Since $m_0 = v_0 = 0$, early estimates are biased toward zero:
  $$\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1 - \beta_2^t}$$
- **Update**:
  $$\theta_{t+1} = \theta_t - \frac{\eta \hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}$$

---


In [ ]:
import numpy as np

class SGDMomentum:
    """SGD with momentum optimizer."""
    def __init__(self, lr=0.01, beta=0.9):
        self.lr = lr
        self.beta = beta
        self.velocities = {}

    def update(self, params, grads):
        for key in params:
            if key not in self.velocities:
                self.velocities[key] = np.zeros_like(params[key])
            self.velocities[key] = self.beta * self.velocities[key] + (1 - self.beta) * grads[key]
            params[key] -= self.lr * self.velocities[key]

class Adam:
    """Adam optimizer with full bias correction."""
    def __init__(self, lr=0.001, beta1=0.9, beta2=0.999, epsilon=1e-8):
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.epsilon = epsilon
        self.m = {}  # First moment estimates
        self.v = {}  # Second moment estimates
        self.t = 0   # Timestep counter

    def update(self, params, grads):
        self.t += 1
        for key in params:
            if key not in self.m:
                self.m[key] = np.zeros_like(params[key])
                self.v[key] = np.zeros_like(params[key])
            # Update biased first moment
            self.m[key] = self.beta1 * self.m[key] + (1 - self.beta1) * grads[key]
            # Update biased second moment
            self.v[key] = self.beta2 * self.v[key] + (1 - self.beta2) * (grads[key] ** 2)
            # Bias correction
            m_hat = self.m[key] / (1 - self.beta1 ** self.t)
            v_hat = self.v[key] / (1 - self.beta2 ** self.t)
            # Parameter update
            params[key] -= self.lr * m_hat / (np.sqrt(v_hat) + self.epsilon)



In [ ]:
# ── Comparing optimizers on Rosenbrock-like surface ──
def rosenbrock(x, y):
    """Rosenbrock function: f(x,y) = (1-x)^2 + 100(y-x^2)^2."""
    return (1 - x)**2 + 100 * (y - x**2)**2

def rosenbrock_grad(x, y):
    """Gradient of Rosenbrock function."""
    dx = -2*(1 - x) + 100 * 2*(y - x**2) * (-2*x)
    dy = 100 * 2*(y - x**2)
    return dx, dy

print("--- Optimizer Comparison on Rosenbrock Surface ---")
for opt_name, opt_class, opt_kwargs in [
    ("SGD+Momentum", SGDMomentum, {"lr": 0.0001, "beta": 0.9}),
    ("Adam",         Adam,        {"lr": 0.01, "beta1": 0.9, "beta2": 0.999}),
]:
    params = {"x": np.array([-1.0]), "y": np.array([1.0])}
    optimizer = opt_class(**opt_kwargs)
    
    for step in range(2000):
        gx, gy = rosenbrock_grad(params["x"][0], params["y"][0])
        grads = {"x": np.array([gx]), "y": np.array([gy])}
        optimizer.update(params, grads)
    
    final_loss = rosenbrock(params["x"][0], params["y"][0])
    print(f"  {opt_name:>14} → x={params['x'][0]:+.4f}, y={params['y'][0]:+.4f}, Loss={final_loss:.6f}")

print("\n[Result] Adam converges much faster to the minimum at (1, 1).")
